# Run MoM Project on Google Colab (T4 GPU)

This is a **separate launcher notebook** for running the existing MoM Streamlit dashboard from Google Drive.

- It uses the project files already stored in `MoM_Project`.
- Keep model caches and temporary files on Colab's local `/content` storage for speed; save project files and outputs in Google Drive.
- Colab `/content` is temporary and is cleared when the runtime is reset/disconnected.

**Before starting:** In Colab, choose **Runtime → Change runtime type → T4 GPU** (if available), then run these cells from top to bottom.

## 1. Mount Google Drive and locate the project

In [1]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os, sys, subprocess, importlib.util, textwrap, time

PROJECT_DIR = Path("/content/drive/MyDrive/MTechIndProj/MoM_Project")
assert PROJECT_DIR.exists(), f"Project folder not found: {PROJECT_DIR}"

DASHBOARD_DIR = PROJECT_DIR / "07_dashboard"
APP_FILE = DASHBOARD_DIR / "app.py"
PIPELINE_FILE = DASHBOARD_DIR / "meeting_pipeline.py"

assert APP_FILE.exists(), f"Missing Streamlit app: {APP_FILE}"
assert PIPELINE_FILE.exists(), f"Missing pipeline: {PIPELINE_FILE}"

os.chdir(DASHBOARD_DIR)
print("Project:", PROJECT_DIR)
print("Dashboard:", DASHBOARD_DIR)
print("Python:", sys.version.split()[0])

Mounted at /content/drive
Project: /content/drive/MyDrive/MTechIndProj/MoM_Project
Dashboard: /content/drive/MyDrive/MTechIndProj/MoM_Project/07_dashboard
Python: 3.13.16


## 2. Confirm the Colab GPU

In [2]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "GPU is not enabled. In Colab choose Runtime → Change runtime type → T4 GPU, "
        "then reconnect and rerun this notebook."
    )

!nvidia-smi

PyTorch: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
Tue Nov 24 04:57:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   57C    P8             11W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |               

## 3. Install the tested demo environment

This cell uses the existing `requirements_demo.txt`. It does not reinstall PyTorch; Colab supplies the CUDA-enabled PyTorch runtime.

If the requirements file is in a different location, update `REQUIREMENTS_FILE` below. Run this cell once per fresh runtime. If Colab asks for a runtime restart after installation, restart and continue from Section 1.

In [3]:
candidates = [
    PROJECT_DIR / "07_dashboard" / "requirements_demo.txt",
    PROJECT_DIR / "requirements_demo.txt",
]
REQUIREMENTS_FILE = next((p for p in candidates if p.exists()), None)

if REQUIREMENTS_FILE is None:
    raise FileNotFoundError(
        "requirements_demo.txt was not found in the project root or 07_dashboard folder."
    )

print("Installing from:", REQUIREMENTS_FILE)
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-r", str(REQUIREMENTS_FILE)
])

Installing from: /content/drive/MyDrive/MTechIndProj/MoM_Project/07_dashboard/requirements_demo.txt


0

In [4]:
%pip install reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 16.5 MB/s  0:00:00


## 4. Load the Hugging Face token securely

In Colab, open the **key icon (Secrets)** in the left sidebar and make sure the existing secret `HF_TOKENnew` is available to this notebook. The token is stripped of accidental whitespace and is never printed.

If you later rename the Colab secret, change `HF_SECRET_NAME` below.

In [5]:
from google.colab import userdata

HF_SECRET_NAME = "HF_TOKENnew"
hf_token = userdata.get(HF_SECRET_NAME)

if not hf_token or not hf_token.strip():
    raise RuntimeError(
        f"Colab Secret '{HF_SECRET_NAME}' is missing or empty. "
        "Add it under the Colab Secrets (key icon) panel and enable notebook access."
    )

os.environ["HF_TOKEN"] = hf_token.strip()
print("Hugging Face token loaded:", bool(os.environ.get("HF_TOKEN")))

Hugging Face token loaded: True


## 5. Run the existing environment checker

The current checker validates package versions, GPU, model access, and core pipeline imports. This cell also checks that the two Phase 2 DAG modules are present and importable.

The DAG check is a module/import check only; it does not run Qwen inference or validate dependencies on a real meeting.

In [6]:
checker_candidates = [
    DASHBOARD_DIR / "demo_environment_check.py",
    PROJECT_DIR / "demo_environment_check.py",
]
CHECKER_FILE = next((p for p in checker_candidates if p.exists()), None)

if CHECKER_FILE is None:
    raise FileNotFoundError("demo_environment_check.py was not found.")

print("Running:", CHECKER_FILE)
subprocess.run([sys.executable, str(CHECKER_FILE)], cwd=str(DASHBOARD_DIR), check=True)

# Phase 2 DAG module checks
import importlib
for module_name in ("task_dependency_dag", "dependency_extraction"):
    module = importlib.import_module(module_name)
    print(f"{module_name}: import OK")

from task_dependency_dag import (
    build_dependency_dag, validate_dependency_dag,
    get_topological_order, get_ready_tasks
)
from dependency_extraction import extract_action_dependency_dag
print("Phase 2 DAG API imports: OK")

Running: /content/drive/MyDrive/MTechIndProj/MoM_Project/07_dashboard/demo_environment_check.py
task_dependency_dag: import OK
dependency_extraction: import OK
Phase 2 DAG API imports: OK


## 6. Set temporary/cache locations on Colab local storage

This avoids using the mounted Drive virtual filesystem for large model-cache writes. These folders are temporary and will need to be recreated in a new runtime.

In [7]:
CACHE_ROOT = Path("/content/mom_cache")
for folder in ("huggingface", "torch", "transformers", "tmp"):
    (CACHE_ROOT / folder).mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(CACHE_ROOT / "huggingface")
os.environ["HUGGINGFACE_HUB_CACHE"] = str(CACHE_ROOT / "huggingface" / "hub")
os.environ["TORCH_HOME"] = str(CACHE_ROOT / "torch")
os.environ["TRANSFORMERS_CACHE"] = str(CACHE_ROOT / "transformers")
os.environ["TMPDIR"] = str(CACHE_ROOT / "tmp")

print("Local cache root:", CACHE_ROOT)
print("Project outputs should remain under:", PROJECT_DIR / "outputs")

Local cache root: /content/mom_cache
Project outputs should remain under: /content/drive/MyDrive/MTechIndProj/MoM_Project/outputs


## 7. Start Streamlit

This starts one Streamlit process on port `8501`. It writes logs to Colab local storage. If the app is already running, stop that process or restart the runtime before rerunning this cell.

In [15]:
%cd /content/drive/MyDrive/MTechIndProj/MoM_Project

!pkill -f "streamlit run"

/content/drive/MyDrive/MTechIndProj/MoM_Project


In [16]:
import subprocess, time
from pathlib import Path

LOG_FILE = Path("/content/mom_streamlit.log")
PORT = 8501

# Stop only an existing Streamlit process serving this app, if present.
subprocess.run(
    "pkill -f 'streamlit run app.py' || true",
    shell=True,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(2)

log_handle = open(LOG_FILE, "w")
streamlit_process = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run", str(APP_FILE),
        "--server.port", str(PORT),
        "--server.address", "0.0.0.0",
        "--server.headless", "true"
    ],
    cwd=str(DASHBOARD_DIR),
    stdout=log_handle,
    stderr=subprocess.STDOUT
)
time.sleep(8)

if streamlit_process.poll() is not None:
    print(LOG_FILE.read_text(errors="ignore")[-6000:])
    raise RuntimeError("Streamlit exited during startup. See the log above.")

print(f"Streamlit process started (PID {streamlit_process.pid}) on port {PORT}.")
print("Recent log:")
print(LOG_FILE.read_text(errors="ignore")[-3000:])

Streamlit process started (PID 20438) on port 8501.
Recent log:


2026-11-24 06:11:21.340 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.16.229.229:8501




## 8. Create a public access URL with Cloudflare Tunnel

Run this cell after Streamlit starts. It downloads `cloudflared` into `/content` and starts a tunnel to port `8501`. Keep this cell running while using the dashboard; the public URL appears in its output.

Do not share the public URL beyond people who are meant to access your demo.

In [18]:
CLOUDFLARED = Path("/content/cloudflared")
if not CLOUDFLARED.exists():
    subprocess.check_call([
        "bash", "-lc",
        "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 "
        "-O /content/cloudflared && chmod +x /content/cloudflared"
    ])

print("Starting Cloudflare Tunnel. Wait for the https://...trycloudflare.com URL.")
tunnel_process = subprocess.Popen(
    [str(CLOUDFLARED), "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

# Print initial tunnel output; keep this cell active for the tunnel session.
for _ in range(80):
    line = tunnel_process.stdout.readline()
    if not line:
        break
    print(line, end="")
    if "trycloudflare.com" in line:
        print("\nOpen the https URL above to access the MoM dashboard.")
        break

Starting Cloudflare Tunnel. Wait for the https://...trycloudflare.com URL.
2026-11-24T06:11:33Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-11-24T06:11:33Z INF Requesting new quick Tunnel on trycloudflare.com...

Open the https URL above to access the MoM dashboard.


In [19]:
import time

for _ in range(30):
    line = tunnel_process.stdout.readline()
    if not line:
        time.sleep(1)
        continue

    print(line, end="")

    if "trycloudflare.com" in line:
        print("\nOpen this HTTPS URL to access your MoM dashboard.")
        break

2026-11-24T06:11:36Z INF +--------------------------------------------------------------------------------------------+
2026-11-24T06:11:36Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-11-24T06:11:36Z INF |  https://cradle-define-corpus-investment.trycloudflare.com                                 |

Open this HTTPS URL to access your MoM dashboard.


If the Streamlit server disconnects ,kill the process and Restart the Streamlit.

In [ ]:
!pkill -f "streamlit run"

In [12]:
!pip install -q edge-tts pydub
!apt-get install -y ffmpeg

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:6.1.1-3ubuntu5).
0 upgraded, 0 newly installed, 0 to remove and 33 not upgraded.


In [13]:
import asyncio
import edge_tts
import os
from pydub import AudioSegment

OUTPUT_DIR = "/content/drive/MyDrive/MTechIndProj/MoM_Project/data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_FILE = os.path.join(
    OUTPUT_DIR,
    "Phase2_Project_Launch_Test_1.mp3"
)

# Three different voices
VOICES = {
    "ALEX": "en-US-GuyNeural",
    "PRIYA": "en-IN-NeerjaNeural",
    "OMAR": "en-US-AriaNeural"
}

dialogue = [
    ("ALEX", "Good morning, everyone. Let's discuss the tasks we need to complete before launching the new checkout feature. We need to agree on the order of work and make sure there are no blockers."),
    ("PRIYA", "Good morning. I've completed most of the payment integration. However, we still need to confirm whether the payment API supports multiple promotional codes in a single transaction."),
    ("ALEX", "That's important. Priya, please test the payment API and confirm whether users can apply multiple promotional codes or only one. We need that information before the checkout UI can be finalized."),
    ("PRIYA", "Sure. I'll test the API today and share the results with the team by tomorrow morning."),
    ("OMAR", "I have already started working on the checkout screen. But I don't want to finalize the promotional-code section until I know what the API supports."),
    ("ALEX", "Correct. Omar, you can continue with the general layout, but you must finalize the checkout UI only after Priya completes the API test and shares the results."),
    ("OMAR", "Understood. Once Priya confirms the API behaviour, I'll update the promotional-code section and complete the checkout UI."),
    ("PRIYA", "One more thing. The API documentation also mentions a possible validation issue when two promotional codes are submitted together. I'll include that in my test results."),
    ("ALEX", "Good. So the first dependency is clear: API testing must be completed before the checkout UI is finalized."),
    ("OMAR", "After I finish the UI, we should ask the QA team to test the complete checkout flow. Otherwise, we may miss integration issues."),
    ("ALEX", "Agreed. Omar, once the UI is finalized, please hand it over to the QA team for regression testing. QA testing must happen after the UI work is complete."),
    ("OMAR", "I'll do that. I'll also document the updated UI behaviour so the QA team knows what to test."),
    ("PRIYA", "What about deployment? Can we schedule it for Friday?"),
    ("ALEX", "We can plan for Friday, but deployment must happen only after QA completes regression testing and provides formal sign-off. We shouldn't deploy before that approval."),
    ("OMAR", "That makes sense. So the order is API testing, UI finalization, QA testing, and then deployment."),
    ("ALEX", "Exactly. I'll coordinate the deployment once QA signs off. Priya, your immediate priority is the API test. Omar, your UI finalization depends on Priya's result."),
    ("PRIYA", "I'll share the API test report tomorrow morning."),
    ("OMAR", "I'll wait for the API confirmation before finalizing the promotional-code section."),
    ("ALEX", "One separate task: I'll prepare the project release notes and share the first draft tomorrow. This can happen independently while the technical tasks are progressing."),
    ("PRIYA", "That shouldn't block the API testing."),
    ("ALEX", "Correct. Let's summarize. Priya completes API testing first. Omar finalizes the checkout UI after receiving Priya's results. QA performs regression testing after the UI is complete. Deployment happens only after QA sign-off. I'll prepare the release notes independently."),
    ("OMAR", "Sounds good. We have a clear sequence now."),
    ("ALEX", "Great. Let's proceed with these tasks and review the progress in our next meeting.")
]

async def generate_audio():
    combined = AudioSegment.empty()

    for index, (speaker, text) in enumerate(dialogue):
        voice = VOICES[speaker]
        temp_file = f"/content/segment_{index}.mp3"

        communicate = edge_tts.Communicate(
            text=text,
            voice=voice,
            rate="-5%"
        )

        await communicate.save(temp_file)

        segment = AudioSegment.from_file(temp_file)
        combined += segment + AudioSegment.silent(duration=550)

        os.remove(temp_file)

    combined.export(
        OUTPUT_FILE,
        format="mp3",
        bitrate="128k"
    )

    print("Audio generated:", OUTPUT_FILE)
    print("Duration:", round(len(combined) / 60000, 2), "minutes")

await generate_audio()

/usr/local/lib/python3.13/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


Audio generated: /content/drive/MyDrive/MTechIndProj/MoM_Project/data/Phase2_Project_Launch_Test_1.mp3
Duration: 4.06 minutes


In [14]:
from IPython.display import Audio, display
import os

assert os.path.exists(OUTPUT_FILE), "Audio file not found"

display(Audio(OUTPUT_FILE))

Output hidden; open in https://colab.research.google.com to view.